# OV-reader eval — single-example walkthrough

Loads 1–2 datapoints for a chosen config, marks the exact pivot tokens, and runs the **same**
pipeline as `scripts/evaluation/eval_ov_reader.py` — GoalScope skyline + `OVReader` across layers,
then the oracle grading (`hard_eval`). Every step calls the script's own functions; only the
`rich` visualization is notebook-specific.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os, sys
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0,1")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.append(".."); sys.path.append("../..")

import torch
from rich import print as rich_print
from rich.markup import escape
from rich.table import Table

# ── reuse the runner + io + goalscope utilities verbatim ──
import scripts.evaluation.eval_ov_reader as R
from scripts.evaluation import ov_reader_io as ovio
from scripts.evaluation.eval_goalscope import hard_eval, CAUSAL_PREFIX
from src.goals.goalscope import OVReader, apply_goalscope, greedy_continue, _build_causality_prompt
from src.tokens import prepare_input
from src.models import ModelandTokenizer

# run from the repo root so relative paths (head_sets/, globals.yml) resolve like the script
# (which runs from the repo root). Idempotent: root derived from R.__file__, not the CWD.
os.chdir(os.path.abspath(os.path.join(os.path.dirname(R.__file__), "..", "..")))

# ── config (matches a run of the script; same SEED -> same datapoints -> oracle cache hits) ──
MODEL         = "Qwen/Qwen2.5-Coder-32B-Instruct"
DATATYPE      = "str"           # int | str | list[int] | list[str]
LANG          = "python"        # python | cpp
MODE          = "causality"     # faithfulness | causality
N_CONDITIONS  = [2, 3, 4]
LAYERS        = [36, 40, 44, 48, 52, 56]     # sparse probe layers
PIVOTS        = ovio.PIVOT_ORDER # ['prev_end','indent','var','eq','first_impl']
ALPHA         = 0.15             # OVReader (additive_norm)
SKYLINE_ALPHA = 0.1              # GoalScope skyline
MAX_NEW       = 64               # read-out length
GEN_NEW       = 64               # frontier-body greedy budget
SEED          = 123
N_EXAMPLES    = 2
HEADS_JSON    = R.DEFAULT_HEADS_JSON.format(slug=MODEL.split("/")[-1])
torch.set_grad_enabled(False)

env.yml not found in /share/NFS/u/sfeucht/goals!
Setting MODEL_ROOT="". Models will now be downloaded to conda env cache, if not already there
Other defaults are set to:
    DATA_DIR = "data"
    RESULTS_DIR = "results"
    HPARAMS_DIR = "hparams"


torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [3]:
# ── load model, the 122 goal heads, and OVReader (readers cached once) ──
mt = ModelandTokenizer(model_key=MODEL, dtype=torch.bfloat16,
                       device_map="auto", attn_implementation="eager", 
                       cache_dir="/share/NFS/models")
n_heads = mt.model.config.num_attention_heads
HEADS = R.load_json_heads(HEADS_JSON, mt.n_layer, n_heads)
ov = OVReader(mt, HEADS)
GOALSCOPE_PROMPT = _build_causality_prompt(mt) if MODE == "causality" else None
PREFIX = CAUSAL_PREFIX if MODE == "causality" else ""
rich_print(f"heads={len(HEADS)}  layers={LAYERS}  mode={MODE}  dtype={DATATYPE}/{LANG}")

Qwen/Qwen2.5-Coder-32B-Instruct not found in 
If not found in cache, model will be downloaded from HuggingFace to cache directory: None


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Building OV readers:   0%|          | 0/122 [00:00<?, ?it/s]

heads=122  layers=[36, 40, 44, 48, 52, 56]  mode=causality  dtype=str/python

In [4]:
# ── build N_EXAMPLES datapoints (prefill-to-frontier) + dark-mode viz helpers ──
from rich.text import Text
from rich.panel import Panel
from rich.console import Group

# pivot palette lifted from notebooks/fig_n_table_generators/eval_goal.ipynb (muted tones; used as
# text HIGHLIGHTS here — bold white on the colour reads well on a dark VS Code background).
ANCHOR_COLOR = {"prev_end": "#b5675f", "indent": "#6b6aa1", "var": "#d1a054",
                "eq": "#a7c48a", "first_impl": "#3f8079", "OVERALL": "#9a9a9a"}
CODE_FG, GEN_FG = "grey74", "grey54"   # prompt vs generated foreground (dark-mode)
def trunc(s, n=90):
    return (s[:n] + "…") if s and len(s) > n else (s or "")

def pivot_text(full_text, pivots, boundary):
    """full_text as a rich Text: pivot tokens highlighted in ANCHOR_COLOR, generated tokens dimmed."""
    ids = prepare_input(prompts=full_text, tokenizer=mt)["input_ids"][0].tolist()
    pos2name = {p: n for n, p in pivots.items()}
    t = Text()
    for i, tid in enumerate(ids):
        s = mt.tokenizer.decode([tid])
        if i in pos2name:
            t.append(s, style=f"bold white on {ANCHOR_COLOR[pos2name[i]]}")
        else:
            t.append(s, style=GEN_FG if i >= boundary else CODE_FG)
    return t

datapoints = R.sample_datapoints(DATATYPE, N_CONDITIONS, 64, LANG, SEED)[:N_EXAMPLES]
examples = []
for i, inst in enumerate(datapoints):
    prompt, var = R.build_prompt(mt, inst, LANG)
    completion = greedy_continue(mt, **prepare_input(prompts=prompt, tokenizer=mt), max_new_tokens=GEN_NEW)
    full_text = prompt + completion
    examples.append(dict(
        spec=f"{DATATYPE}_{LANG}_{i:03d}", inst=inst, prompt=prompt, completion=completion,
        full_text=full_text, var=var, pivots=R.pivot_positions(mt, prompt, full_text, PIVOTS),
        boundary=int(prepare_input(prompts=prompt, tokenizer=mt)["input_ids"].shape[1]),
        impl_span=R.frontier_impl_span(prompt, completion, var), frontier_desc=inst.expected_next_desc))
print(f"built {len(examples)} datapoints for {DATATYPE}/{LANG}")

built 2 datapoints for str/python


In [5]:
# ── full prompt + generation, with the pivot tokens highlighted (eval_goal.ipynb palette) ──
legend = Text("pivots:  ", style="bold")
for n in PIVOTS:
    legend.append(f" {n} ", style=f"bold white on {ANCHOR_COLOR[n]}")
    legend.append("   ")
blocks = [legend]
for e in examples:
    meta = Text.assemble(
        ("frontier   ", "bold"), (e["frontier_desc"], f"bold {ANCHOR_COLOR['first_impl']}"),
        (f"       T={e['inst'].meta['T']}  j={e['inst'].meta['j']}  var={e['var']}", "grey62"),
        ("\nimpl_span  ", "bold"), (e["impl_span"], ANCHOR_COLOR["var"]))
    blocks.append(Panel(Group(meta, Text(""), pivot_text(e["full_text"], e["pivots"], e["boundary"])),
                        title=f"[bold]{e['spec']}[/]", title_align="left", border_style="grey42"))
rich_print(Group(*blocks))

pivots:   prev_end     indent     var     eq     first_impl    
╭─ str_python_000 ────────────────────────────────────────────────────────────────────────────────────────────────╮
│ frontier   the length of the string       T=2  j=1  var=b                                                       │
│ impl_span  b = len(s)                                                                                           │
│                                                                                                                 │
│ <|im_start|>system                                                                                              │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>                                                              │
│ <|im_start|>user                                                                                                │
│ Write a Python program that implements a function `func` which, given a string `s`, computes in sequential      │
│ order:                                                                                                          │
│ (1) the number of uppercase letters                                                                             │
│ (2) the length of the string                                                                                    │
│ <|im_end|>                                                                                                      │
│ <|im_start|>assistant                                                                                           │
│ <code>                                                                                                          │
│ def func(s: str):                                                                                               │
│     a = sum(1 for c in s if c.isupper())                                                                        │
│     b = len(s)                                                                                                  │
│     return a, b                                                                                                 │
│ </code>                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─ str_python_001 ────────────────────────────────────────────────────────────────────────────────────────────────╮
│ frontier   whether the first character is a letter       T=3  j=1  var=b                                        │
│ impl_span  b = s[0].isalpha() if s else False                                                                   │
│                                                                                                                 │
│ <|im_start|>system                                                                                              │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>                                                              │
│ <|im_start|>user                                                                                                │
│ Write a Python program that implements a function `func` which, given a string `s`, computes in sequential      │
│ order:                                                                                                          │
│ (1) whether the string contains a digit                                                                         │
│ (2) whether the first character is a letter                                                                     │
│ (3) the last character                                                                                    

In [6]:
# ── SKYLINE: goal-head OUTPUT read-out per pivot (apply_goalscope, additive_norm) ──
for e in examples:
    apos = sorted(set(e["pivots"].values()))
    res = apply_goalscope(e["full_text"], mt=mt, selected_heads=HEADS, strategy="additive_norm",
                          steer_layer_range=R.STEER_LAYER_RANGE, alpha=SKYLINE_ALPHA,
                          n_steer_tokens="all", max_new_tokens=MAX_NEW, show=False,
                          include_noncode=False, token_positions=apos, goalscope_prompt=GOALSCOPE_PROMPT)
    by = {r["pos"]: r for r in res}
    e["skyline"] = {piv: PREFIX + by.get(pos, {}).get("patched_gen", "") for piv, pos in e["pivots"].items()}

sky = Table(title="GoalScope skyline — goal-head OUTPUT read-out", title_style="bold",
            header_style="bold", border_style="grey42")
sky.add_column("pivot", no_wrap=True)
for e in examples:
    sky.add_column(e["spec"], overflow="fold", max_width=64)
for piv in PIVOTS:
    if any(piv in e["pivots"] for e in examples):
        sky.add_row(f"[{ANCHOR_COLOR[piv]}]{piv}[/]",
                    *[escape(e["skyline"].get(piv, "")) for e in examples])
rich_print(sky)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

                                   GoalScope skyline — goal-head OUTPUT read-out                                   
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ pivot      ┃ str_python_000                                   ┃ str_python_001                                  ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ prev_end   │ bool check(auto value){ return (value > 0) &&    │ bool check(auto value){ return (value > 0) &&   │
│            │ (value < 100); }                                 │ (value % 2 == 0); }                             │
│ indent     │ bool check(auto value){ return (value > 0); }    │ bool check(auto value){ return value > 0 &&     │
│            │                                                  │ isalpha(value) && isupper(value); }             │
│ var        │ bool check(auto value){ return value > 0; }      │ bool check(auto value){ return isalpha(value)   │
│            │                                                  │ && value > 'a' - 1; }                           │
│ eq         │ bool check(auto value){ return value.size(); }   │ bool check(auto value){ return                  │
│            │                                                  │ isalpha(value[0]); }                            │
│ first_impl │ bool check(auto value){ return value.size(); }   │ bool check(auto value){ return                  │
│            │                                                  │ isalpha(value[0]); }                            │
└────────────┴──────────────────────────────────────────────────┴─────────────────────────────────────────────────┘

In [7]:
# ── OVReader: OV-write read-out per pivot across LAYERS (one batched call per pivot) ──
for e in examples:
    e["ov"] = {}
    for piv, pos in e["pivots"].items():
        by = ov.apply_probe_on_prompt(e["full_text"], pos, LAYERS, mode=MODE,
                            steer_layers=list(range(*R.STEER_LAYER_RANGE)),
                            max_new_tokens=MAX_NEW, alpha=ALPHA)
        e["ov"][piv] = {L: PREFIX + ro for L, ro in by.items()}

tables = []
for e in examples:
    t = Table(title=f"OVReader · {e['spec']} · {e['frontier_desc']}",
              title_style=f"bold {ANCHOR_COLOR['first_impl']}", header_style="bold", border_style="grey42")
    t.add_column("pivot", no_wrap=True)
    for L in LAYERS:
        t.add_column(f"L{L}", overflow="fold", max_width=26)
    for piv in PIVOTS:
        if piv in e["ov"]:
            t.add_row(f"[{ANCHOR_COLOR[piv]}]{piv}[/]",
                      *[escape(trunc(e["ov"][piv][L])) for L in LAYERS])
    tables.append(t)
rich_print(Group(*tables))

                               OVReader · str_python_000 · the length of the string                                
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ pivot      ┃ L36            ┃ L40            ┃ L44            ┃ L48            ┃ L52            ┃ L56           ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ prev_end   │ bool           │ bool           │ bool           │ bool           │ bool           │ bool          │
│            │ check(auto     │ check(auto     │ check(auto     │ check(auto     │ check(auto     │ check(auto    │
│            │ value){        │ value){        │ value){ return │ value){ return │ value){ return │ value){       │
│            │     int count  │     int count  │ value; }       │ (value == 10); │ value == 0; }  │ return        │
│            │ = 0;           │ = 0;           │ int count = 0; │ }              │                │ value.size(); │
│            │     for(auto&  │     for(int i  │ for(int i = 0; │                │                │ }             │
│            │ i : value){    │ = 0; i <=      │ i < 100; ++i){ │                │                │               │
│            │         if(i   │ value; i++){   │     i…         │                │                │               │
│            │ >= 'a')…       │     …          │                │                │                │               │
│ indent     │ bool           │ bool           │ bool           │ bool           │ bool           │ bool          │
│            │ check(auto     │ check(auto     │ check(auto     │ check(auto     │ check(auto     │ check(auto    │
│            │ value){ return │ value){ return │ value){ return │ value){ return │ value){ return │ value){       │
│            │ value > 0; }   │ value > 0; }   │ value > 0; }   │ sizeof(value)  │ value == 0; }  │ return value  │
│            │                │                │                │ < 2; }         │                │ > 0; }        │
│ var        │ bool           │ bool           │ bool           │ bool           │ bool           │ bool          │
│            │ check(auto     │ check(auto     │ check(auto     │ check(auto     │ check(auto     │ check(auto    │
│            │ value){ return │ value){ return │ value){ return │ value){ return │ value){ return │ value){       │
│            │ value; }       │ value; }       │ value; }       │ sizeof(value); │ value.size();  │ return        │
│            │                │                │                │ }              │ }              │ value.size(); │
│            │                │                │                │                │                │ }             │
│ eq         │ bool           │ bool           │ bool           │ bool           │ bool           │ bool          │
│            │ check(auto     │ check(auto     │ check(auto     │ check(auto     │ check(auto     │ check(auto    │
│            │ value){ return │ value){ return │ value){ return │ value){ return │ value){ return │ value){       │
│            │ value.size();  │ value.size();  │ sizeof(value); │ value.size();  │ value.size();  │ return        │
│            │ }              │ }              │ }              │ }              │ }              │ value.size(); │
│            │                │                │                │                │                │ }             │
│ first_impl │ bool           │ bool           │ bool           │ bool           │ bool           │ bool          │
│            │ check(auto     │ check(auto     │ check(auto     │ check(auto     │ check(auto     │ check(auto    │
│            │ value){ return │ value){ return │ value){ return │ value){ return │ value){ return │ value){       │
│            │ value == 0; }  │ value.size();  │ value.size();  │ value.size();  │ value.size();  │ return        │
│            │                │ }              │ }      

In [8]:
# ── FULL EVAL: grade every read-out (hard_eval); dark-mode verdict tables + hard_yes summary ──
VC = {"YES": "bold green3", "NO": "bold red3", "NO_INFO": "yellow3",
      "HEAVY_LOOP": "magenta3", "MALFORMED": "grey50", "UNPARSED": "grey50"}
def vtag(v):
    return f"[{VC.get(v, 'white')}]{v}[/]"

records, verdict_tables = [], []
for e in examples:
    t = Table(title=f"verdicts · {e['spec']}", title_style="bold", header_style="bold",
              border_style="grey42")
    t.add_column("pivot", no_wrap=True)
    t.add_column("goalscope", no_wrap=True)
    for L in LAYERS:
        t.add_column(f"L{L}", no_wrap=True)
    for piv in PIVOTS:
        if piv not in e["pivots"]:
            continue
        sv = hard_eval(e["skyline"][piv], e["impl_span"], MODE)["verdict"]
        records.append(dict(spec=e["spec"], pivot=piv, layer_idx="goalscope", hard=sv))
        row = [f"[{ANCHOR_COLOR[piv]}]{piv}[/]", vtag(sv)]
        for L in LAYERS:
            v = hard_eval(e["ov"][piv][L], e["impl_span"], MODE)["verdict"]
            records.append(dict(spec=e["spec"], pivot=piv, layer_idx=L, hard=v))
            row.append(vtag(v))
        t.add_row(*row)
    verdict_tables.append(t)

# hard_yes rollup via the SAME util the runner uses (ovio.compute_summary)
summ = Table(title="hard_yes summary — same compute_summary as the runner", title_style="bold",
             header_style="bold", border_style="grey42")
summ.add_column("layer", no_wrap=True)
for piv in PIVOTS:
    summ.add_column(piv, no_wrap=True)
summ.add_column("OVERALL", no_wrap=True)
for idx in ["goalscope"] + LAYERS:
    recs = [r for r in records if r["layer_idx"] == idx]
    s = ovio.compute_summary(recs, n_samples=len(examples))
    cells = [("—" if (hy := s["by_pivot"].get(piv, {}).get("hard_yes")) is None
              else f"[{ANCHOR_COLOR[piv]}]{hy:.2f}[/]") for piv in PIVOTS]
    summ.add_row(str(idx), *cells, f"[bold]{s['overall']['hard_yes']:.2f}[/]")
rich_print(Group(*verdict_tables, Text(""), summ))

env.yml not found in /share/NFS/u/sfeucht/goals!


UnboundLocalError: cannot access local variable 'config' where it is not associated with a value